> **Copie exécutée de référence.** Ce fichier est une exécution complète de [`tutorial.ipynb`](tutorial.ipynb), conservée telle quelle pour montrer à quoi ressemble une exécution réussie (sorties de cellules incluses) — y compris les cellules dont les commandes réelles (SOMEF, gh, archivage SWH) restent volontairement commentées faute d'URL de fork réelle. Pour suivre le tutoriel vous-même, ouvrez et exécutez `tutorial.ipynb`, pas cette copie.

# Tutorial : décrire, préserver, identifier et citer un logiciel de recherche

Version exécutable du parcours du webinaire **« Citer et préserver des codes et logiciels
avec SWHID et CodeMeta »** (15 octobre 2026). Ce notebook reprend
[`docs/guided-demo.md`](../docs/guided-demo.md) et
[`docs/getting-started.md`](../docs/getting-started.md) sous forme de cellules exécutables :
suivez-le du haut vers le bas sur votre propre fork de ce dépôt.

**Fil rouge :** le [projet QGIS](../qgis/rewilding_portugal_species_2025.qgz) de suivi
d'espèces de la Vallée du Côa (Portugal).

> **Human-in-the-loop, dès le départ.** Ce notebook — comme les métadonnées QGIS et le
> `codemeta.json` de ce dépôt — a été produit avec l'assistance d'un agent de codage IA
> (Claude Code, via le serveur MCP QGIS pour les appels PyQGIS), sous la direction et la
> relecture de Linda Angulo Lopez. Chaque étape ci-dessous est faite pour être **exécutée et
> vérifiée par vous**, pas simplement copiée : l'assistance d'un agent ne dispense pas de
> comprendre ce qui se passe à chaque étape (voir la Section 8 pour pourquoi cela compte aussi
> pour vos propres métadonnées).

## Section 0 — Préparation

Vérifiez que vous disposez de :
- `git`, et un fork cloné de ce dépôt ;
- [QGIS](https://qgis.org/) 3.34+ avec les bindings Python (PyQGIS) ;
- Python 3.12 avec `somef`, `codemetapy`, `requests` installés (`pip install somef codemetapy requests`) ;
- (optionnel) le [GitHub CLI](https://cli.github.com/) `gh`, authentifié, pour la Section 4.

In [1]:
import shutil, sys, subprocess

checks = {
    "git": shutil.which("git"),
    "gh (optionnel)": shutil.which("gh"),
}
for name, path in checks.items():
    print(f"{name}: {'OK — ' + path if path else 'introuvable'}")

# Le paquet PyPI s'appelle "codemetapy" (pip install codemetapy) mais s'importe
# sous le nom "codemeta" — d'où la paire (nom pip, nom import) ci-dessous.
for pip_name, import_name in [("somef", "somef"), ("codemetapy", "codemeta"), ("requests", "requests")]:
    try:
        __import__(import_name)
        print(f"python package {pip_name}: OK")
    except ImportError:
        print(f"python package {pip_name}: MANQUANT — pip install {pip_name}")

try:
    from qgis.core import Qgis
    print("PyQGIS: OK —", Qgis.QGIS_VERSION)
except ImportError:
    print("PyQGIS: introuvable dans ce noyau Python.")
    print("  -> Ouvrez ce notebook depuis la console Python de QGIS, ou un environnement")
    print("     où qgis.core est importable, pour exécuter les Sections 1 et 3.")

git: OK — /usr/bin/git
gh (optionnel): OK — /usr/bin/gh
python package somef: OK
python package codemetapy: OK
python package requests: OK


PyQGIS: OK — 3.34.4-Prizren


## Section 1 — Ouvrir le projet QGIS avec PyQGIS

On ouvre le projet directement depuis votre fork fraîchement cloné, pour confirmer que le
travail de portabilité (données copiées dans `data/`, chemins relatifs re-pointés) a bien
fonctionné — sans dépendre d'un quelconque autre dépôt sur votre machine.

In [2]:
from qgis.core import QgsApplication, QgsProject
import os

# Ajustez si votre notebook n'est pas exécuté depuis notebooks/
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
QGS_PATH = os.path.join(REPO_ROOT, "qgis", "rewilding_portugal_species_2025.qgs")

# Si aucune instance QGIS n'est déjà active (notebook lancé hors QGIS), on en crée une
# minimale, sans interface graphique, uniquement pour charger le projet.
if not QgsApplication.instance():
    qgs = QgsApplication([], False)
    qgs.initQgis()

project = QgsProject.instance()
ok = project.read(QGS_PATH)
print("Chargé :", ok, "-", QGS_PATH)
print("Nombre de couches :", len(project.mapLayers()))

QSocketNotifier: Can only be used with threads started with QThread


Chargé : True - /home/linda/citer-preserver-logiciels/qgis/rewilding_portugal_species_2025.qgs
Nombre de couches : 7


Warning 4: Failed to open /usr/share/qgis/resources/data/world_map.gpkg: Permission non accordée.


In [3]:
for layer_id, layer in project.mapLayers().items():
    print(f"{layer.name():55s} | valide={layer.isValid()!s:5s} | source={layer.source()[:70]}")

Aggregate species counts (not itemised in report)       | valide=True  | source=file:///home/linda/citer-preserver-logiciels/data/annual_review_2025_a
Côa river                                               | valide=True  | source=/home/linda/citer-preserver-logiciels/data/study_area.gpkg|layername=c
Carte du monde                                          | valide=True  | source=/usr/share/qgis/resources/data/world_map.gpkg|layername=countries
Greater Côa Valley study area (30km)                    | valide=True  | source=/home/linda/citer-preserver-logiciels/data/study_area.gpkg|layername=s
Named species present (Greater Côa Valley)              | valide=True  | source=file:///home/linda/citer-preserver-logiciels/data/annual_review_2025_s
Species — Faia Brava, Vale Carapito, Ermo das Águias, Ribeira do Mosteiro | valide=True  | source=file:///home/linda/citer-preserver-logiciels/data/visitor_board_specie
OpenStreetMap                                           | valide=True  | source=c

## Section 2 — Décrire le logiciel avec CodeMeta

Le `codemeta.json` à la racine du dépôt est l'exemple de référence. Générez le vôtre avec
**SOMEF** sur votre fork (voir aussi
[`exercises/generate-codemeta.md`](../exercises/generate-codemeta.md) pour la version
GitHub Actions de cet exercice), puis comparez.

In [4]:
import json

with open(os.path.join(REPO_ROOT, "codemeta.json")) as f:
    reference_codemeta = json.load(f)

print(json.dumps(reference_codemeta, indent=2, ensure_ascii=False))

{
  "@context": "https://doi.org/10.5063/schema/codemeta-2.0",
  "@type": "SoftwareSourceCode",
  "name": "Citer et préserver des codes et logiciels avec SWHID et CodeMeta",
  "description": "Worked example repository pairing a plain GitHub Pages site with a real QGIS project (Greater Côa Valley rewilding species monitoring) for the 2026-10-15 webinar on describing, preserving, identifying and citing research software with CodeMeta, Software Heritage and SWHID. Adapted from the RSECon26 workshop 'Making Research Software FAIR with CodeMeta'.",
  "codeRepository": "https://github.com/lindangulopez/citer-preserver-logiciels",
  "license": "https://spdx.org/licenses/MIT",
  "programmingLanguage": "HTML",
  "runtimePlatform": "QGIS 3.34",
  "keywords": [
    "GIS",
    "QGIS",
    "rewilding",
    "ecological connectivity",
    "SWHID",
    "Software Heritage",
    "CodeMeta",
    "open science",
    "research software citation"
  ],
  "author": [
    {
      "@type": "Person",
      "give

In [5]:
import subprocess

# Remplacez par l'URL de VOTRE fork avant d'exécuter.
FORK_URL = "https://github.com/<vous>/citer-preserver-logiciels"

# D'abord une seule fois : somef configure --auto (télécharge les modèles NLTK nécessaires).
# Vérifié contre `somef describe --help` (somef 0.11.3) : -c/--codemeta_out prend un CHEMIN
# en argument (ce n'est pas un simple drapeau booléen — une version précédente de ce notebook
# et du workflow GitHub Actions se trompait sur ce point, corrigé après un vrai run raté).
# --github-token évite la limite de ~60 requêtes/heure de l'API GitHub sans authentification.
cmd = [
    "somef", "describe", "-r", FORK_URL,
    "-c", "codemeta.generated.json", "-t", "0.8", "-p",
    "--github-token", "VOTRE_TOKEN_GITHUB",
]
print("Commande à exécuter (décommentez la ligne suivante quand FORK_URL et le token sont renseignés) :")
print(" ".join(cmd))
# subprocess.run(cmd, check=True)

Commande à exécuter (décommentez la ligne suivante quand FORK_URL et le token sont renseignés) :
somef describe -r https://github.com/<vous>/citer-preserver-logiciels -c codemeta.generated.json -t 0.8 -p --github-token VOTRE_TOKEN_GITHUB


In [6]:
# Une fois généré, comparez avec le fichier de référence :
generated_path = "codemeta.generated.json"
if os.path.exists(generated_path):
    with open(generated_path) as f:
        generated = json.load(f)
    ref_keys = set(reference_codemeta) - {"_comment"}
    gen_keys = set(generated)
    print("Champs présents seulement dans la référence :", ref_keys - gen_keys)
    print("Champs présents seulement dans le généré     :", gen_keys - ref_keys)
    print("Champs communs                                :", ref_keys & gen_keys)
else:
    print(f"{generated_path} n'existe pas encore — exécutez la cellule précédente d'abord.")

codemeta.generated.json n'existe pas encore — exécutez la cellule précédente d'abord.


## Section 3 — Métadonnées QGIS natives (projet + par couche)

`codemeta.json` décrit le **logiciel**. Le projet QGIS porte, lui, ses propres métadonnées
natives (`QgsProjectMetadata` / `QgsLayerMetadata`), qui décrivent les **données** — un exemple
concret de deux couches de métadonnées complémentaires, pas redondantes.

Le projet livré dans ce dépôt a déjà ces métadonnées renseignées, comme exemple de référence.
À vous de les relire, puis de les modifier vous-même sur votre fork.

In [7]:
# Lecture des métadonnées de référence (déjà présentes dans le projet livré)
md = project.metadata()
print("Titre    :", md.title())
print("Résumé   :", md.abstract()[:150], "...")
print("Auteur   :", md.author())
print("Mots-clés:", dict(md.keywords()))
print("Liens    :", [(l.name, l.url) for l in md.links()])

Titre    : Rewilding Portugal — Greater Côa Valley species monitoring (teaching example)
Résumé   : QGIS project pairing a 30km Greater Côa Valley study-area boundary with named-site species observations and visitor-board sightings, repurposed as the ...
Auteur   : Linda Angulo Lopez
Mots-clés: {'gis-teaching': ['rewilding', 'ecological connectivity', 'Côa Valley', 'species monitoring', 'SWHID', 'CodeMeta']}
Liens    : [('codemeta.json', 'https://github.com/lindangulopez/citer-preserver-logiciels/blob/main/codemeta.json')]


In [8]:
layer_name_by_id = {lid: lyr.name() for lid, lyr in project.mapLayers().items()}
for lid, lyr in project.mapLayers().items():
    lmd = lyr.metadata()
    if lmd.abstract():
        print(f"- {lyr.name()}")
        print(f"    abstract: {lmd.abstract()[:90]}...")
        print(f"    keywords: {dict(lmd.keywords())}")
        print(f"    history : {lmd.history()}")

- Aggregate species counts (not itemised in report)
    abstract: Non-spatial table of aggregate species counts (not itemised in the public report) from the...
    keywords: {'gis-teaching': ['species counts', 'annual review', 'Côa Valley']}
    history : ["Repurposed for teaching from Linda Angulo Lopez's private rewilding-illustration research project (Solo-field-illustrator-project-for-Rewilding-Portugal); file paths were re-pointed into this repository's data/ folder for self-containment."]
- Côa river
    abstract: Côa river mainstem line geometry, part of the study-area GeoPackage....
    keywords: {'gis-teaching': ['hydrology', 'Côa Valley']}
    history : ["Repurposed for teaching from Linda Angulo Lopez's private rewilding-illustration research project (Solo-field-illustrator-project-for-Rewilding-Portugal); file paths were re-pointed into this repository's data/ folder for self-containment."]
- Greater Côa Valley study area (30km)
    abstract: 30 km buffer study-area boundar

In [9]:
# À VOUS : modifiez les métadonnées sur VOTRE fork (exemple sur la couche study_area).
# Adaptez l'auteur, le résumé, les mots-clés à votre propre travail si vous réutilisez ce
# projet pour autre chose.
from qgis.core import QgsLayerMetadata

STUDY_AREA_LAYER_NAME = "Greater Côa Valley study area (30km)"
target = next((l for l in project.mapLayers().values() if l.name() == STUDY_AREA_LAYER_NAME), None)

if target is not None:
    new_md = QgsLayerMetadata()
    new_md.setAbstract("VOTRE description ici — qu'est-ce que cette couche représente pour vous ?")
    new_md.setKeywords({"gis-teaching": ["mon-mot-cle-1", "mon-mot-cle-2"]})
    contact = QgsLayerMetadata.Contact("VOTRE NOM")
    contact.role = "author"
    new_md.setContacts([contact])
    new_md.setHistory(["Modifié dans le cadre du tutoriel SWHID/CodeMeta, à partir du projet original."])
    target.setMetadata(new_md)
    print("Métadonnées mises à jour pour :", target.name())
    print("Nouveau résumé :", target.metadata().abstract())
else:
    print(f"Couche {STUDY_AREA_LAYER_NAME!r} introuvable — vérifiez le nom exact ci-dessus.")

Métadonnées mises à jour pour : Greater Côa Valley study area (30km)
Nouveau résumé : VOTRE description ici — qu'est-ce que cette couche représente pour vous ?


In [10]:
# Sauvegardez sur VOTRE copie (ne modifiez pas le fichier du dépôt original par erreur !).
save_path = os.path.join(REPO_ROOT, "qgis", "rewilding_portugal_species_2025.my-edits.qgs")
project.write(save_path)
print("Sauvegardé :", save_path)

# Rechargez pour confirmer la persistance :
reload_project = QgsProject()
reload_project.read(save_path)
reloaded_layer = next((l for l in reload_project.mapLayers().values() if l.name() == STUDY_AREA_LAYER_NAME), None)
print("Après rechargement, abstract :", reloaded_layer.metadata().abstract() if reloaded_layer else "couche introuvable")

Sauvegardé : /home/linda/citer-preserver-logiciels/qgis/rewilding_portugal_species_2025.my-edits.qgs
Après rechargement, abstract : VOTRE description ici — qu'est-ce que cette couche représente pour vous ?


ERROR 1: sqlite3_exec(DROP TRIGGER "rtree_countries_geom_update3") failed: attempt to write a readonly database
ERROR 1: sqlite3_exec(CREATE TRIGGER "rtree_countries_geom_update3" AFTER UPDATE ON "countries" WHEN OLD."fid" != NEW."fid" AND (NEW."geom" NOTNULL AND NOT ST_IsEmpty(NEW."geom")) BEGIN DELETE FROM "rtree_countries_geom" WHERE id = OLD."fid"; INSERT OR REPLACE INTO "rtree_countries_geom" VALUES (NEW."fid",ST_MinX(NEW."geom"), ST_MaxX(NEW."geom"),ST_MinY(NEW."geom"), ST_MaxY(NEW."geom")); END) failed: trigger "rtree_countries_geom_update3" already exists
ERROR 1: sqlite3_exec(DROP TRIGGER "rtree_states_provinces_geom_update3") failed: attempt to write a readonly database
ERROR 1: sqlite3_exec(CREATE TRIGGER "rtree_states_provinces_geom_update3" AFTER UPDATE ON "states_provinces" WHEN OLD."fid" != NEW."fid" AND (NEW."geom" NOTNULL AND NOT ST_IsEmpty(NEW."geom")) BEGIN DELETE FROM "rtree_states_provinces_geom" WHERE id = OLD."fid"; INSERT OR REPLACE INTO "rtree_states_provinces_

## Section 4 — GitHub Actions

Sur votre fork, déclenchez et inspectez les workflows fournis :
`.github/workflows/generate-codemeta.yml` et `validate-codemeta.yml`.

In [11]:
import shutil, subprocess

FORK_REPO = "<vous>/citer-preserver-logiciels"  # remplacez par votre fork

if shutil.which("gh"):
    print("Déclenchement manuel du workflow (nécessite `gh auth login` au préalable) :")
    cmd = ["gh", "workflow", "run", "generate-codemeta.yml", "--repo", FORK_REPO]
    print(" ".join(cmd))
    # subprocess.run(cmd, check=True)

    print()
    print("Suivi des exécutions récentes :")
    cmd2 = ["gh", "run", "list", "--repo", FORK_REPO, "--workflow=generate-codemeta.yml", "--limit", "5"]
    print(" ".join(cmd2))
    # subprocess.run(cmd2, check=True)
else:
    print("gh CLI non trouvé — déclenchez le workflow manuellement depuis l'onglet")
    print("Actions de votre fork sur GitHub (workflow_dispatch).")

Déclenchement manuel du workflow (nécessite `gh auth login` au préalable) :
gh workflow run generate-codemeta.yml --repo <vous>/citer-preserver-logiciels

Suivi des exécutions récentes :
gh run list --repo <vous>/citer-preserver-logiciels --workflow=generate-codemeta.yml --limit 5


## Section 5 — Software Heritage & SWHID

L'archivage se fait via un formulaire web — ce n'est pas scriptable sans jeton d'API dédié
(hors périmètre de ce tutoriel). Une fois votre fork archivé, en revanche, on peut interroger
l'API publique de Software Heritage pour retrouver et afficher son SWHID.

**Étape manuelle :** allez sur
[archive.softwareheritage.org/save](https://archive.softwareheritage.org/save/) et soumettez
l'URL de votre fork.

In [12]:
import requests

FORK_URL = "https://github.com/<vous>/citer-preserver-logiciels"  # remplacez par votre fork

# API publique Software Heritage : statut d'une origine
api_url = f"https://archive.softwareheritage.org/api/1/origin/{FORK_URL}/get/"
try:
    resp = requests.get(api_url, timeout=10)
    if resp.status_code == 200:
        print(json.dumps(resp.json(), indent=2))
    else:
        print(f"Statut {resp.status_code} — le dépôt n'est peut-être pas encore archivé.")
        print("Soumettez-le d'abord via https://archive.softwareheritage.org/save/")
except requests.RequestException as e:
    print("Requête échouée (pas de connexion ?) :", e)

Statut 404 — le dépôt n'est peut-être pas encore archivé.
Soumettez-le d'abord via https://archive.softwareheritage.org/save/


> **Exemple de référence :** une fois `lindangulopez/citer-preserver-logiciels` archivé, ses
> SWHID seront ajoutés ici et dans [`docs/guided-demo.md`](../docs/guided-demo.md), pour que
> vous ayez un exemple concret de la forme exacte d'un SWHID (`swh:1:snp:...`, `swh:1:rev:...`).

## Section 6 — Citation : de `codemeta.json` à `CITATION.cff`, Zenodo, HAL

Un `CITATION.cff` minimal peut être dérivé directement des champs déjà présents dans
`codemeta.json` — la correspondance est en grande partie directe.

In [13]:
def codemeta_to_cff_stub(codemeta: dict) -> str:
    authors = codemeta.get("author", [])
    if isinstance(authors, dict):
        authors = [authors]
    cff_authors = "\n".join(
        f"  - family-names: {a.get('familyName', '')}\n    given-names: {a.get('givenName', '')}"
        for a in authors
    )
    return f"""cff-version: 1.2.0
message: "Si vous utilisez ce logiciel, merci de le citer comme suit."
title: "{codemeta.get('name', '')}"
authors:
{cff_authors}
repository-code: "{codemeta.get('codeRepository', '')}"
license: "{codemeta.get('license', '').rstrip('/').split('/')[-1] if codemeta.get('license') else ''}"
"""

print(codemeta_to_cff_stub(reference_codemeta))

cff-version: 1.2.0
message: "Si vous utilisez ce logiciel, merci de le citer comme suit."
title: "Citer et préserver des codes et logiciels avec SWHID et CodeMeta"
authors:
  - family-names: Angulo Lopez
    given-names: Linda
repository-code: "https://github.com/lindangulopez/citer-preserver-logiciels"
license: "MIT"



**Zenodo** : lier le dépôt GitHub à Zenodo permet d'obtenir un DOI à chaque release. Le DOI et
le SWHID répondent à des questions différentes — le DOI est ce que l'on cite dans une
bibliographie, le SWHID est ce qui permet de vérifier précisément quel code se trouve derrière
cette citation.

**HAL** : mentionné ici comme un exemple parmi d'autres, particulièrement pertinent dans le
contexte français et pour la curation par les bibliothécaires — ce n'est pas le sujet principal
de ce webinaire (voir [`docs/guided-demo.md`](../docs/guided-demo.md) Partie 4).

## Section 7 — Vous avez terminé quand...

- [ ] Le projet QGIS de votre fork s'ouvre avec les 7 couches valides (Section 1).
- [ ] Vous avez généré et relu votre propre `codemeta.json` (Section 2).
- [ ] Vous avez lu ET modifié vous-même les métadonnées QGIS (projet + au moins une couche) sur votre copie (Section 3).
- [ ] Les workflows `generate-codemeta.yml` et `validate-codemeta.yml` ont tourné sur votre fork (Section 4).
- [ ] Votre fork est archivé dans Software Heritage et vous avez récupéré son SWHID (Section 5).
- [ ] Vous pouvez expliquer en une phrase ce que décrit CodeMeta, ce qu'identifie un SWHID, et ce qu'apporte un DOI (Zenodo) ou une notice HAL (Section 6).

## Section 8 — Créditer les agents de codage IA

Cette section s'appuie sur le cours [*Agentic Coding for Geospatial*](https://spatialthoughts.com/courses/agentic-coding-geospatial/)
(Spatial Thoughts), construit autour de Claude Code piloté via des serveurs MCP pour des tâches
géospatiales concrètes — avec une exigence centrale de **validation humaine à chaque étape**
(« human-in-the-loop ») plutôt qu'une confiance aveugle dans la sortie de l'agent. L'exemple mis
en avant par ce cours est parlant : un agent avait correctement diagnostiqué qu'une image avait
rétréci sous la taille de travail attendue par un modèle après un ré-échantillonnage, et corrigé
le pipeline en conséquence — plutôt que de masquer le symptôme.

**Ce dépôt applique cette éthique, pas seulement en parle :** les métadonnées QGIS (Section 3),
le `codemeta.json` de référence, et une bonne partie de ce notebook ont été produits avec
l'assistance de Claude Code (via le serveur MCP QGIS pour les appels PyQGIS), sous la direction
et la relecture de Linda Angulo Lopez à chaque étape.

**Comment créditer l'assistance d'un agent dans VOTRE fork :**

1. **Dans vos commits**, comme le fait ce dépôt lui-même : une ligne `Co-Authored-By:` en pied
   de message de commit lorsqu'un agent a produit ou modifié significativement le contenu.
2. **Dans vos métadonnées** : CodeMeta n'a pas de champ dédié « assisté par IA ». Deux options
   honnêtes existent — utiliser `contributor` (distinct d'`author`) pour signaler l'outillage,
   ou documenter cela simplement dans un `NOTICE.md`/README. Ni l'une ni l'autre n'est
   « la » bonne réponse — c'est la même ambiguïté de modélisation que celle déjà rencontrée en
   Section 2 pour « un projet QGIS comme partie du logiciel ». Faites un choix, documentez-le.

**À retenir :** si le logiciel est un résultat de recherche qui doit être décrit, préservé,
identifié et citable, le même raisonnement s'applique à la **façon dont il a été produit** —
l'intervention d'un agent fait partie de cette provenance, elle ne s'en distingue pas.